# FastICA Audio Source Separation — reproducible research code

**Author and code maintainer:** **Md. Razu Ahmed** · [ORCID](https://orcid.org/0009-0001-8646-1881) · [Email](mailto:razuahmed038@gmail.com).  
**Published article:** Ahmed, M. R., Mauya, J., Reza, M. S., & Amin, R. (2026). *Sophisticated Audio Source Separation: A Statistical Exploration of Clarity and Precision With FastICA*. *Engineering Reports, 8*(1), e70575. [https://doi.org/10.1002/eng2.70575](https://doi.org/10.1002/eng2.70575).

## Choose an execution profile

- **`original` (default):** the paper's private three MP3 files, with original FastICA/PCA/NMF settings; **requires authorized local access**. Not included in this repository.
- **`demo`:** first run `python scripts/generate_demo_audio.py`, set `FASTICA_PROFILE=demo` and `FASTICA_RUNS=1`, then run all notebook cells. The included generator creates deterministic **synthetic signals, not human speech**.
- **`custom`:** set `FASTICA_PROFILE=custom` and provide **three distinct audio files** in the private `Raw Data/` folder named `source_1.wav`, `source_2.wav`, and `source_3.wav`, or edit `CUSTOM_AUDIO_FILENAMES` in the configuration cell. Alternatively supply full paths through `FASTICA_AUDIO_FILES`. New files can be mono or stereo and may differ in duration or sampling rate; all are converted to mono, resampled to 48 kHz, trimmed, RMS-normalized and cropped to the shortest source.

**Playback enabled on your local machine:** `ENABLE_INLINE_AUDIO_PLAYBACK=True`. Clear all executed notebook cell outputs before committing, because Jupyter embeds audio. Private WAV export and speech-signal plots remain disabled by default. The code does not upload data anywhere.

### Scientific scope

The method constructs a fixed-seed 3×3 **instantaneous linear mixture** from three known reference audio sources, applies FastICA and PCA to multichannel mixtures and NMF to a mono-STFT mixture, and evaluates against the known references using BSS Eval with best-permutation alignment. The scores published in the article apply to its **original** recordings and settings; new recordings produce new results and may behave differently. No real-world separation of an unknown, recorded single-channel mixture is claimed.

### Setup and evidence

Install Python 3.12, FFmpeg (`ffmpeg` on PATH), and `python -m pip install -r requirements.txt` from the repository root. See [`README.md`](../README.md), [`REPRODUCIBILITY.md`](../REPRODUCIBILITY.md), [`DATA_AVAILABILITY.md`](../DATA_AVAILABILITY.md), and [`LICENSE`](../LICENSE). This notebook was reconstructed from the user's original Jupyter HTML export and safety/portability improvements have been made. The full original ten-run result has not been independently recomputed in this release audit.


## Imports

In [ ]:
"""
Blind Source Separation (FastICA + PCA + NMF) with reproducible evaluation
- Canonicalizes audio (mp3->wav), logs raw channels/duration/sr, then mono+trim.
- Builds synthetic mixtures X = A @ S from aligned sources (crop_min).
- Baselines: FastICA (multichannel), PCA (multichannel), NMF (mono-STFT with soft masks).
- K repeated runs; metrics via mir_eval.bss_eval_sources; report mean ± SD.
- Saves CSV/JSON artifacts and figures; progress via tqdm; logs env + GPU.
- Handles ICA permutation by aligning estimates to references. The original
  published speaker-variant display order is used only in the original profile.

Author: (Md. Razu Ahmed)
"""

# ============== Imports ==============
import os, sys, json, csv, platform, warnings, subprocess, shutil, time
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict

import numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.display
import soundfile as sf
from scipy.signal import butter, filtfilt
from sklearn.decomposition import FastICA, PCA, NMF
from mir_eval.separation import bss_eval_sources
from IPython.display import Audio, display

warnings.filterwarnings("ignore", category=UserWarning)

# tqdm (progress bars)
try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **k): return x  # fallback no-op


## Configuration

In [ ]:
# ============== Dataset and portable paths ==============
from pathlib import Path

# Choose "original" (published private recordings), "custom" (your own 3 recordings),
# or "demo" (locally generated, non-speech synthetic WAVs).  An environment
# variable overrides this setting, e.g. FASTICA_PROFILE=demo.
PROFILE = os.environ.get("FASTICA_PROFILE", "original").strip().lower()
if PROFILE not in {"original", "custom", "demo"}:
    raise ValueError("FASTICA_PROFILE must be one of: original, custom, demo")

# Work whether Jupyter was launched from the repository root or notebooks/.
WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = next((d for d in (WORKING_DIR, *WORKING_DIR.parents)
                  if (d / "requirements.txt").is_file() and (d / "notebooks").is_dir()), WORKING_DIR)
DEFAULT_PRIVATE_DIR = (REPO_ROOT / "Raw Data")
FILE_DIR = Path(os.environ.get("FASTICA_PRIVATE_AUDIO_DIR", str(DEFAULT_PRIVATE_DIR))).expanduser().resolve()

# For custom runs edit these filenames (one distinct source in each file).
# WAV, MP3, FLAC, OGG and other FFmpeg-decodable formats are supported.
CUSTOM_AUDIO_FILENAMES = ["source_1.wav", "source_2.wav", "source_3.wav"]
CUSTOM_SOURCE_LABELS = ["Source 1", "Source 2", "Source 3"]

if PROFILE == "original":
    INPUT_FILENAMES = [
        "Sample recording 1 (Male+Male).mp3",
        "Sample recording 2 (Female+Female).mp3",
        "Sample recording 3 (Male+Female).mp3",
    ]
    SOURCE_SHORT = ["Male+Male", "Female+Female", "Male+Female"]
    # Display mapping from the published study's alignment: ref 3, ref 1, ref 2.
    SEPARATED_SHORT = ["Male+Female", "Male+Male", "Female+Female"]
elif PROFILE == "demo":
    FILE_DIR = (REPO_ROOT / "demo_audio").resolve()
    INPUT_FILENAMES = ["source_1.wav", "source_2.wav", "source_3.wav"]
    SOURCE_SHORT = ["Synthetic 1", "Synthetic 2", "Synthetic 3"]
    SEPARATED_SHORT = SOURCE_SHORT.copy()
else:
    # Optional: supply absolute filenames via FASTICA_AUDIO_FILES, separated by
    # os.pathsep (';' on Windows, ':' on macOS/Linux), without editing the notebook.
    raw_files = os.environ.get("FASTICA_AUDIO_FILES", "")
    INPUT_FILENAMES = ([x.strip() for x in raw_files.split(os.pathsep) if x.strip()]
                       if raw_files else list(CUSTOM_AUDIO_FILENAMES))
    SOURCE_SHORT = list(CUSTOM_SOURCE_LABELS)
    SEPARATED_SHORT = SOURCE_SHORT.copy()

if len(INPUT_FILENAMES) != 3 or len(SOURCE_SHORT) != 3:
    raise ValueError("This 3-source study requires exactly three input files and three labels")
if len(set(SOURCE_SHORT)) != 3:
    raise ValueError("SOURCE_SHORT labels must be unique")
FILE_PATHS = [str(Path(name).expanduser().resolve() if Path(name).is_absolute()
                  else (FILE_DIR / name).resolve()) for name in INPUT_FILENAMES]
if len(set(FILE_PATHS)) != 3:
    raise ValueError("Please supply three DISTINCT audio files")

WAVE_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']
LABEL_COLOR = dict(zip(SOURCE_SHORT, WAVE_COLORS))
SEP_WAVE_COLORS = ['#9467bd', '#8c564b', '#17becf']
SEP_LABEL_COLOR = dict(zip(SEPARATED_SHORT, SEP_WAVE_COLORS))

OUTPUT_DIR = str(REPO_ROOT / "outputs_bss")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input profile: {PROFILE} | Three audio files | Results: {OUTPUT_DIR}")


## Logging helper (tee to file)

In [ ]:
# ============== Logging helper (tee to file) ==============
class TeeLogger:
    def __init__(self, path):
        self.path = path
        self._fh = open(path, "w", encoding="utf-8")
    def close(self):
        self._fh.close()
    def log(self, msg="", end="\n"):
        print(msg, end=end)
        self._fh.write(msg + ("" if end == "" else "\n"))
        self._fh.flush()

LOG_PATH = os.path.join(OUTPUT_DIR, "results_report.txt")
LOGGER = TeeLogger(LOG_PATH)


## Dataclasses

In [ ]:
# ============== Dataclasses ==============
@dataclass
class STFTParams:
    n_fft: int = 2048
    hop: int = 512
    win: int = 2048
    window: str = "hann"
    db_floor: float = -80.0

@dataclass
class PreprocessParams:
    target_sr: int | None = 48000    # align sample rates; original data already 48 kHz
    trim_top_db: float = 25.0
    bandpass_hz: Tuple[int, int] | None = None  # e.g., (300, 3400) or None
    filter_order: int = 5
    normalize_rms: bool = True
    apply_lowpass_dereverb: bool = False
    lowpass_cut_hz: int = 3000
    apply_spectral_sub: bool = False
    noise_percentile: float = 0.1      # 0..1; for simple spectral subtraction

@dataclass
class MixParams:
    mix_seed: int = 0
    add_noise_snr_db: float | None = None

@dataclass
class FastICAParams:
    n_components: int | None = None
    algorithm: str = "parallel"        # "parallel" or "deflation"
    fun: str = "logcosh"               # "logcosh" | "exp" | "cube"
    max_iter: int = 1000
    tol: float = 1e-4
    whiten: str | bool = "unit-variance"  # "unit-variance" | "arbitrary-variance" | False

@dataclass
class PCANMFParams:
    pca_solver: str = "full"           # "full" | "randomized"
    nmf_max_iter: int = 500
    nmf_tol: float = 1e-4
    nmf_beta_loss: str = "kullback-leibler"  # "kullback-leibler" | "frobenius"
    nmf_init: str = "nndsvda"
    nmf_solver: str = "mu"             # "mu" needed for KL loss

@dataclass
class ExperimentParams:
    runs: int = 10
    save_wavs_for_run: int | None = 0  # display/play aligned sources for run index; WAV export controlled separately
    save_figs: bool = True

STFT_CFG = STFTParams()
PREP_CFG = PreprocessParams(bandpass_hz=None)
MIX_CFG  = MixParams()
ICA_CFG  = FastICAParams()
BASE_CFG = PCANMFParams()
EXP_CFG  = ExperimentParams()
# For a fast first demonstration, run one repetition: FASTICA_RUNS=1.
# Default remains ten runs as in the paper.
EXP_CFG.runs = int(os.environ.get("FASTICA_RUNS", str(EXP_CFG.runs)))
if EXP_CFG.runs < 1: raise ValueError("FASTICA_RUNS must be a positive integer")
# Local audio playback is enabled when you have the authorized recordings.
# Audio remains private only if notebook outputs are cleared before any public commit.
ENABLE_INLINE_AUDIO_PLAYBACK = True
# Public-release safeguards. Do not write or embed participant speech by default.
# The public notebook must be re-cleared after executing playback.

# These switches only control presentation/export, not the separation calculations.
SAVE_PRIVATE_AUDIO_FILES = False
SHOW_PRIVATE_SIGNAL_FIGURES = False


## Device / Environment logging

In [ ]:
# ============== Device / Environment logging ==============
def detect_device_info() -> Dict[str, str]:
    info = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "librosa": librosa.__version__,
        "matplotlib": plt.matplotlib.__version__,
    }
    try:
        import sklearn, mir_eval
        info["scikit_learn"] = sklearn.__version__
        info["mir_eval"] = mir_eval.__version__
    except Exception:
        pass
    try:
        import torch
        info["torch"] = torch.__version__
        if torch.cuda.is_available():
            dev = torch.cuda.current_device()
            info["device"] = f"cuda:{dev}"
            info["gpu_name"] = torch.cuda.get_device_name(dev)
            info["gpu_capability"] = str(torch.cuda.get_device_capability(dev))
            info["gpu_mem_total_GB"] = f"{torch.cuda.get_device_properties(dev).total_memory / (1024**3):.2f}"
        else:
            info["device"] = "cpu"
    except Exception:
        info["device"] = "cpu"
    return info

def save_json(path: str, obj: Dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def save_csv(path: str, rows: List[Dict[str, float]], fieldnames: List[str]):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames); w.writeheader(); w.writerows(rows)

def ensure_dir(p: str): os.makedirs(p, exist_ok=True)


## Canonicalization & raw info

In [ ]:
# ============== Canonicalization & raw info ==============
def ffmpeg_exists() -> bool:
    return shutil.which("ffmpeg") is not None

def canonicalize_to_wav(path: str) -> str:
    root, ext = os.path.splitext(path)
    if ext.lower() == ".wav":
        return path
    # Local WAV cache lives beside the private audio, not in the repository.
    # Reject an existing same-stem WAV: never overwrite another source file.
    out_wav = root + ".canonicalized.wav"
    # Rebuild a cached WAV when its input is newer; avoid silently using stale data.
    if not os.path.exists(out_wav) or os.path.getmtime(out_wav) < os.path.getmtime(path):
        if not ffmpeg_exists():
            raise RuntimeError("ffmpeg not found on PATH; cannot canonicalize to WAV.")
        cmd = ["ffmpeg", "-y", "-i", path, out_wav]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        LOGGER.log(f"[Canonicalized] {os.path.basename(path)} → {os.path.basename(out_wav)} (ffmpeg)")
    return out_wav

def read_raw_info(path_wav: str) -> Dict:
    with sf.SoundFile(path_wav, 'r') as f:
        channels = f.channels
        sr = f.samplerate
        frames = len(f)
        duration = frames / sr
        shape = (frames,) if channels == 1 else (channels, frames)
    return {"format": ".wav", "channels": channels, "sr": sr, "frames": frames,
            "duration": duration, "shape": shape}


## Stats

In [ ]:
# ============== Stats ==============
from scipy.stats import skew, kurtosis, entropy as shannon_entropy

def histogram_entropy(y: np.ndarray, bins: int = 256) -> float:
    hist, edges = np.histogram(y, bins=bins, density=True)
    probs = hist * np.diff(edges)
    probs = np.clip(probs, 1e-12, None); probs /= probs.sum()
    return float(shannon_entropy(probs, base=np.e))

def descriptive_stats(y: np.ndarray, sr: int) -> Dict[str, float]:
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y).flatten()))
    return {
        "mean": float(np.mean(y)),
        "std": float(np.std(y)),
        "min": float(np.min(y)),
        "max": float(np.max(y)),
        "rms": float(np.sqrt(np.mean(y**2))),
        "energy": float(np.sum(y**2)),
        "zcr": zcr
    }

def high_order_stats(y: np.ndarray) -> Dict[str, float]:
    return {"skewness": float(skew(y)), "kurtosis": float(kurtosis(y))}

def print_signal_stats(y: np.ndarray, sr: int, name: str):
    LOGGER.log(f"Statistics for {name}:")
    LOGGER.log(f"Shape: {y.shape}")
    LOGGER.log(f"Duration: {len(y)/sr:.2f} seconds")
    LOGGER.log(f"Sample Rate: {sr} Hz")
    LOGGER.log(f"Number of Channels (after mono): 1")
    LOGGER.log(f"Histogram-Based Entropy: {histogram_entropy(y):.4f}")
    d = descriptive_stats(y, sr)
    LOGGER.log(f"Mean: {d['mean']:.4f}")
    LOGGER.log(f"Standard Deviation: {d['std']:.4f}")
    LOGGER.log(f"Min Value: {d['min']:.4f}")
    LOGGER.log(f"Max Value: {d['max']:.4f}")
    LOGGER.log(f"RMS: {d['rms']:.4f}")
    LOGGER.log(f"Energy: {d['energy']:.4f}")
    LOGGER.log(f"Zero-Crossing Rate: {d['zcr']:.4f}")
    h = high_order_stats(y)
    LOGGER.log(f"Skewness: {h['skewness']:.4f}")
    LOGGER.log(f"Kurtosis: {h['kurtosis']:.4f}")


## Plot helpers

In [ ]:
# ============== Plot helpers ==============
def plot_waveform(y: np.ndarray, sr: int, title: str, color: str, label: str | None = None):
    plt.figure(figsize=(12, 4))
    t = np.arange(len(y)) / sr
    plt.plot(t, y, color=color, label=label if label else title)
    plt.xlabel("Time [s]"); plt.ylabel("Amplitude")
    plt.title(title)
    if label: plt.legend(loc="upper right")
    plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
    plt.tight_layout()
    plt.show()
    plt.close()  # Release large, high-resolution waveform figures.

def plot_spectrogram(y: np.ndarray, sr: int, title: str, stft: STFTParams, outpath: str | None = None):
    Z = librosa.stft(y, n_fft=stft.n_fft, hop_length=stft.hop, win_length=stft.win, window=stft.window)
    D = librosa.amplitude_to_db(np.abs(Z), ref=np.max)
    fig, ax = plt.subplots(figsize=(10, 3.3))
    im = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', vmin=stft.db_floor, vmax=0, ax=ax)
    cbar = plt.colorbar(im, format='%+2.0f dB', ax=ax); cbar.set_label('dB')
    ax.set_title(title)
    plt.tight_layout()
    if outpath: plt.savefig(outpath, dpi=200)
    plt.show()
    plt.close(fig)

def barplot_compare(stats_dict: Dict[str, Dict[str, Tuple[float, float]]], title: str, outpath: str | None = None):
    methods = list(stats_dict.keys())
    metrics = ["SDR", "SIR", "SAR"]
    x = np.arange(len(methods)); width = 0.25
    fig, ax = plt.subplots(figsize=(9,6))
    for i, metric in enumerate(metrics):
        means = [stats_dict[m][metric][0] for m in methods]
        stds  = [stats_dict[m][metric][1] for m in methods]
        bars = ax.bar(x + (i-1)*width, means, width, yerr=stds, capsize=4, label=metric)
        for b in bars:
            h = b.get_height()
            ax.annotate(f"{h:.2f}", xy=(b.get_x()+b.get_width()/2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(methods)
    ax.set_ylabel("Score (dB)"); ax.set_title(title)
    ax.legend(); ax.grid(True, axis='y', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    if outpath: plt.savefig(outpath, dpi=200)
    plt.show()
    plt.close(fig)


## Preprocessing

In [ ]:
# ============== Preprocessing ==============
def bandpass(y: np.ndarray, sr: int, lo: int, hi: int, order: int = 5) -> np.ndarray:
    nyq = 0.5 * sr; lo_n = max(lo / nyq, 1e-4); hi_n = min(hi / nyq, 0.999)
    b, a = butter(order, [lo_n, hi_n], btype="band")
    return filtfilt(b, a, y).astype(np.float32)

def lowpass(y: np.ndarray, sr: int, cut_hz: int, order: int = 4) -> np.ndarray:
    nyq = 0.5 * sr; wn = min(cut_hz / nyq, 0.999)
    b, a = butter(order, wn, btype="low")
    return filtfilt(b, a, y).astype(np.float32)

def spectral_subtraction_simple(y: np.ndarray, n_fft: int, hop: int, win: int, window: str, noise_pct: float) -> np.ndarray:
    Z = librosa.stft(y, n_fft=n_fft, hop_length=hop, win_length=win, window=window)
    Mag = np.abs(Z); Phase = np.angle(Z)
    noise = np.percentile(Mag, q=noise_pct*100.0, axis=1, keepdims=True)
    Mag_est = np.maximum(Mag - noise, 0.0)
    Z_denoised = Mag_est * np.exp(1j*Phase)
    y_hat = librosa.istft(Z_denoised, hop_length=hop, win_length=win, window=window, length=len(y))
    return y_hat.astype(np.float32)

def load_and_preprocess(path: str, cfg: PreprocessParams, stft: STFTParams) -> Tuple[np.ndarray, int]:
    y, sr = librosa.load(path, sr=cfg.target_sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=cfg.trim_top_db)
    if cfg.normalize_rms:
        rms = np.sqrt(np.mean(y**2)) + 1e-12; y = y / rms
    if cfg.apply_lowpass_dereverb:
        y = lowpass(y, sr, cfg.lowpass_cut_hz)
    if cfg.bandpass_hz is not None:
        y = bandpass(y, sr, cfg.bandpass_hz[0], cfg.bandpass_hz[1], cfg.filter_order)
    if cfg.apply_spectral_sub:
        y = spectral_subtraction_simple(y, stft.n_fft, stft.hop, stft.win, stft.window, cfg.noise_percentile)
    return y.astype(np.float32), sr

def preprocess_sources(paths: List[str], cfg: PreprocessParams, stft: STFTParams) -> Tuple[np.ndarray, int, List[str]]:
    LOGGER.log("[INFO] Using files:")
    for p in paths: LOGGER.log(f"    {os.path.basename(p)}")
    LOGGER.log(f"[LOG] Full text report: {LOG_PATH}")

    env = detect_device_info()
    LOGGER.log(f"[Device] {env.get('device','cpu')}  |  GPU: {env.get('gpu_name','N/A')}")
    LOGGER.log(f"[Env] python={env.get('python')} | platform={env.get('platform')} | numpy={env.get('numpy')} | "
               f"sklearn={env.get('scikit_learn','?')} | librosa={env.get('librosa')} | mir_eval={env.get('mir_eval','?')} | "
               f"torch={env.get('torch','?')} | device={env.get('device')} | gpu={env.get('gpu_name','N/A')}")
    LOGGER.log(f"[FastICA params] algorithm={ICA_CFG.algorithm} | fun={ICA_CFG.fun} | max_iter={ICA_CFG.max_iter} | "
               f"tol={ICA_CFG.tol} | whiten={ICA_CFG.whiten}")

    wavs, raw_infos, after_infos, wav_paths = [], [], [], []
    for p in paths:
        wav = canonicalize_to_wav(p); wav_paths.append(wav)
        raw = read_raw_info(wav)
        LOGGER.log(f"[Raw] {os.path.basename(wav)} | format={raw['format']} | channels={raw['channels']} | "
                   f"sr={raw['sr']} Hz | duration={raw['duration']:.2f}s | shape={raw['shape']}")
        y, sr = load_and_preprocess(wav, cfg, stft)
        LOGGER.log(f"[After mono+trim] {os.path.basename(wav)} | duration={len(y)/sr:.2f}s | samples={len(y)}")
        raw_infos.append(raw); after_infos.append({"duration": len(y)/sr, "samples": len(y), "sr": sr})
        wavs.append(y)

    # Reject missing/empty/silent input before ICA and BSS evaluation.
    if len(wavs) != 3:
        raise ValueError("This experiment requires exactly three recordings")
    for source_no, y in enumerate(wavs, 1):
        if y.size < stft.n_fft * 2:
            raise ValueError(f"Source {source_no} is too short after trimming: "
                             f"{y.size} samples; provide at least {stft.n_fft*2}")
        if not np.all(np.isfinite(y)):
            raise ValueError(f"Source {source_no} contains non-finite samples")
        if float(np.std(y)) < 1e-6:
            raise ValueError(f"Source {source_no} is silent or nearly constant")

    # Crop alignment assumes all recordings share the same sample rate.
    # Without this check, equal sample counts could represent different times.
    rates = {item["sr"] for item in after_infos}
    if len(rates) != 1:
        raise ValueError(
            f"Inconsistent sample rates after loading: {sorted(rates)}. "
            "Resample files to a common rate or set PREP_CFG.target_sr."
        )

    # Align to minimum length (crop_min)
    T = min(len(y) for y in wavs)
    wavs = [y[:T] for y in wavs]
    # Almost-identical source tracks make blind separation ill-conditioned.
    normalized = np.stack(wavs, axis=0).astype(np.float64)
    relative_rank = np.linalg.matrix_rank(normalized @ normalized.T, tol=1e-7)
    if relative_rank < 3:
        raise ValueError("Audio sources are not linearly independent: provide 3 distinct tracks")
    LOGGER.log(f"[Align] alignment=crop_min → T={T} samples ({T/after_infos[0]['sr']:.2f}s)")

    # Save dataset overview CSV
    rows = []
    for i, wav in enumerate(wav_paths):
        rows.append({
            "file": os.path.basename(wav_paths[i]),
            "raw_channels": raw_infos[i]["channels"],
            "raw_sr": raw_infos[i]["sr"],
            "raw_duration_sec": raw_infos[i]["duration"],
            "post_mono_trim_duration_sec": after_infos[i]["duration"],
            "aligned_duration_sec": T/after_infos[i]["sr"]
        })
    ds_csv = os.path.join(OUTPUT_DIR, "dataset_overview.csv")
    save_csv(ds_csv, rows, fieldnames=list(rows[0].keys()))
    LOGGER.log(f"[Saved] dataset overview → {ds_csv}")

    LOGGER.log(f"[Dataset] recordings={len(wavs)} | samples_per_track={T} | duration_per_track={T/sr:.2f}s | sr={sr} Hz")
    return np.stack(wavs, axis=0), sr, wav_paths


## Mixing

In [ ]:
# ============== Mixing ==============
def make_mixtures(S_true: np.ndarray, mix_seed: int, add_noise_snr_db: float | None) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(mix_seed)
    S = S_true.shape[0]
    A = rng.normal(size=(S, S))
    A = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    X = A @ S_true
    if add_noise_snr_db is not None:
        sig_pow = np.mean(X**2)
        noise_pow = sig_pow / (10 ** (add_noise_snr_db / 10))
        noise = rng.normal(scale=np.sqrt(noise_pow), size=X.shape)
        X = X + noise
    return X.astype(np.float32), A.astype(np.float32)


## Methods & Alignment

In [ ]:
# ============== Methods & Alignment ==============
def run_fastica(X: np.ndarray, cfg: FastICAParams, seed: int, n_components: int) -> Tuple[np.ndarray, dict]:
    X_T = X.T
    ica = FastICA(n_components=n_components, algorithm=cfg.algorithm, fun=cfg.fun,
                  max_iter=cfg.max_iter, tol=cfg.tol, whiten=cfg.whiten, random_state=seed)
    t0 = time.time()
    S_hat_T = ica.fit_transform(X_T)
    dt = time.time() - t0
    meta = {
        "n_iter": getattr(ica, "n_iter_", None),
        "time": dt,
        "mixing_shape": getattr(ica, "mixing_", np.empty((0,0))).shape,
        "unmixing_shape": getattr(ica, "components_", np.empty((0,0))).shape,
    }
    LOGGER.log(f"[FastICA] n_iter={meta['n_iter']}  time={dt:.3f}s  mixing:{meta['mixing_shape']} unmixing:{meta['unmixing_shape']}")
    return S_hat_T.T.astype(np.float32), meta

def run_pca(X: np.ndarray, n_components: int, seed: int, solver: str) -> Tuple[np.ndarray, dict]:
    X_T = X.T
    pca = PCA(n_components=n_components, svd_solver=solver, whiten=False, random_state=seed)
    t0 = time.time()
    S_pca_T = pca.fit_transform(X_T)
    dt = time.time() - t0
    LOGGER.log(f"[PCA] time={dt:.3f}s")
    LOGGER.log(f"[PCA] explained_variance_ratio: " + ", ".join(f"{x:.3f}" for x in pca.explained_variance_ratio_))
    return S_pca_T.T.astype(np.float32), {"time": dt, "evr": pca.explained_variance_ratio_.tolist()}

def run_nmf(y_mix: np.ndarray, n_components: int, seed: int, stft: STFTParams, base_cfg: PCANMFParams, pb_desc: str = "") -> Tuple[np.ndarray, dict]:
    Z = librosa.stft(y_mix, n_fft=stft.n_fft, hop_length=stft.hop, win_length=stft.win, window=stft.window)
    V = np.abs(Z) + 1e-10
    nmf = NMF(n_components=n_components, init=base_cfg.nmf_init,
              solver=base_cfg.nmf_solver, beta_loss=base_cfg.nmf_beta_loss,
              max_iter=base_cfg.nmf_max_iter, tol=base_cfg.nmf_tol, random_state=seed)
    t0 = time.time()
    W = nmf.fit_transform(V)
    H = nmf.components_
    dt = time.time() - t0
    recon_err = nmf.reconstruction_err_
    LOGGER.log(f"[NMF] n_iter={getattr(nmf, 'n_iter_', '?')}  recon_err={recon_err:.4f}  time={dt:.3f}s  W:{W.shape} H:{H.shape}")
    WH = W @ H + 1e-12
    sources = []
    for k in tqdm(range(n_components), desc=f"NMF masks {pb_desc}", leave=False):
        Mk = (np.outer(W[:, k], H[k, :])) / WH
        Zk = Mk * Z
        yk = librosa.istft(Zk, hop_length=stft.hop, win_length=stft.win, window=stft.window, length=y_mix.shape[0])
        sources.append(yk.astype(np.float32))
    return np.stack(sources, axis=0), {"time": dt, "recon_err": float(recon_err), "n_iter": getattr(nmf, "n_iter_", None)}

def evaluate_and_align(S_true: np.ndarray, S_est: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns (sdr, sir, sar, perm, S_est_aligned_to_true_order)
    """
    T = min(S_true.shape[1], S_est.shape[1])
    sdr, sir, sar, perm = bss_eval_sources(S_true[:, :T], S_est[:, :T])
    S_aligned = S_est[perm]
    return sdr, sir, sar, perm, S_aligned


## Audio I/O & Aggregation

In [ ]:
# ============== Audio I/O & Aggregation ==============
def save_wav(path: str, y: np.ndarray, sr: int):
    y = y / (np.max(np.abs(y)) + 1e-12)
    sf.write(path, (y * 32767.0).astype(np.int16), sr, subtype="PCM_16")
    LOGGER.log(f"Saved audio to {os.path.basename(path)}")

def play_audio(y: np.ndarray, sr: int, label: str):
    if not ENABLE_INLINE_AUDIO_PLAYBACK:
        LOGGER.log(f"[Privacy] Inline audio playback disabled: {label}")
        return
    LOGGER.log(f"[Play] {label}")
    try:
        display(Audio(y, rate=sr))
    except Exception:
        pass

def aggregate_mean_sd(metric_runs: List[Tuple[np.ndarray, np.ndarray, np.ndarray]]) -> Dict[str, Tuple[float, float]]:
    # run-level means over sources, then mean±sd over runs
    sdrs = np.array([np.mean(m[0]) for m in metric_runs])
    sirs = np.array([np.mean(m[1]) for m in metric_runs])
    sars = np.array([np.mean(m[2]) for m in metric_runs])
    return {"SDR": (float(np.mean(sdrs)), float(np.std(sdrs))),
            "SIR": (float(np.mean(sirs)), float(np.std(sirs))),
            "SAR": (float(np.mean(sars)), float(np.std(sars)))}

# Helper: indices to display separated outputs in requested order
def display_indices_from_labels(source_short: List[str], desired_order: List[str]) -> List[int]:
    label_to_idx = {lab: i for i, lab in enumerate(source_short)}
    return [label_to_idx[lab] for lab in desired_order]


## Main

In [ ]:
# ============== Main ==============
def main():
    ensure_dir(OUTPUT_DIR)
    missing = [path for path in FILE_PATHS if not os.path.isfile(path)]
    if missing:
        raise FileNotFoundError(
            "Private audio recordings are required locally and are not in the public notebook. "
            "Use FASTICA_PROFILE=demo after running scripts/generate_demo_audio.py; "
            "for custom audio use FASTICA_PROFILE=custom and three local audio files; "
            "for original study use the authorized recordings in Raw Data/. "
            "Missing: " + ", ".join(os.path.basename(p) for p in missing)
        )

    # 0) Load & preprocess sources with raw reporting
    S_true, sr, wav_paths = preprocess_sources(FILE_PATHS, PREP_CFG, STFT_CFG)
    S, T = S_true.shape
    n_components = ICA_CFG.n_components or S
    disp_order = display_indices_from_labels(SOURCE_SHORT, SEPARATED_SHORT)  # original: [2,0,1]; custom/demo: [0,1,2]
    if n_components != S:
        raise ValueError("For BSS Eval alignment, n_components must equal the three true source tracks")

    # 1) Create mixture
    X, A = make_mixtures(S_true, mix_seed=MIX_CFG.mix_seed, add_noise_snr_db=MIX_CFG.add_noise_snr_db)
    np.savetxt(os.path.join(OUTPUT_DIR, "mixing_matrix_A.csv"), A, delimiter=",")
    LOGGER.log("[Mixing Matrix A] rows=mixture channels; cols=source gains:")
    for r in A:
        LOGGER.log("  " + "  ".join(f"{v:+.4f}" for v in r))

    # folders for audio artifacts
    wav_dir = os.path.join(OUTPUT_DIR, "wavs"); ensure_dir(wav_dir)

    # 2) Save/plot originals (post-trim mono) + mixed mono preview
    for i in range(S):
        if SAVE_PRIVATE_AUDIO_FILES:
            save_wav(os.path.join(wav_dir, f"processed_audio_{i+1}.wav"), S_true[i], sr)
        pretty = f"Sample Recording {i+1} ({SOURCE_SHORT[i]})"
        print_signal_stats(S_true[i], sr, pretty)
        play_audio(S_true[i], sr, pretty)
        if SHOW_PRIVATE_SIGNAL_FIGURES:
            plot_waveform(S_true[i], sr, f"Waveform of {pretty}", LABEL_COLOR[SOURCE_SHORT[i]], label=pretty)
            plot_spectrogram(S_true[i], sr, f"Spectrogram of {pretty}", STFT_CFG)

    y_mix = np.mean(X, axis=0)  # mono preview for plots / NMF
    if SAVE_PRIVATE_AUDIO_FILES:
        save_wav(os.path.join(wav_dir, f"mixed_audio.wav"), y_mix, sr)
    print_signal_stats(y_mix, sr, "Mixed Audio")
    play_audio(y_mix, sr, "Mixed Audio")
    if SHOW_PRIVATE_SIGNAL_FIGURES:
        plot_waveform(y_mix, sr, "Waveform of Mixed Audio", color="#d62728", label="Mixed Audio")
        plot_spectrogram(y_mix, sr, "Spectrogram of Mixed Audio", STFT_CFG)

    # 3) Repeated runs (FastICA, PCA, NMF)
    methods = ["FastICA", "PCA", "NMF"]
    metrics_runs = {m: [] for m in methods}
    per_run_rows = {m: [] for m in methods}

    for run_idx in tqdm(range(EXP_CFG.runs), desc="Runs", total=EXP_CFG.runs):
        seed = run_idx
        LOGGER.log(f"\n--- Run {run_idx}/{EXP_CFG.runs-1} (seed={seed}) ---")

        # FastICA (align to reference)
        S_ica_raw, _meta = run_fastica(X, ICA_CFG, seed=seed, n_components=n_components)
        sdr, sir, sar, perm, S_ica_aligned = evaluate_and_align(S_true, S_ica_raw)
        LOGGER.log("[FastICA] SDR: " + ", ".join(f"{v:.2f}" for v in sdr))
        LOGGER.log("[FastICA] SIR: " + ", ".join(f"{v:.2f}" for v in sir))
        LOGGER.log("[FastICA] SAR: " + ", ".join(f"{v:.2f}" for v in sar))
        metrics_runs["FastICA"].append((sdr, sir, sar))
        for s_idx in range(S):
            per_run_rows["FastICA"].append({"run": run_idx, "source": s_idx+1, "SDR": float(sdr[s_idx]), "SIR": float(sir[s_idx]), "SAR": float(sar[s_idx])})

        # PCA (align to reference)
        S_pca_raw, _ = run_pca(X, n_components=n_components, seed=seed, solver=BASE_CFG.pca_solver)
        sdr_p, sir_p, sar_p, perm_p, S_pca_aligned = evaluate_and_align(S_true, S_pca_raw)
        LOGGER.log("[PCA] SDR: " + ", ".join(f"{v:.2f}" for v in sdr_p))
        LOGGER.log("[PCA] SIR: " + ", ".join(f"{v:.2f}" for v in sir_p))
        LOGGER.log("[PCA] SAR: " + ", ".join(f"{v:.2f}" for v in sar_p))
        metrics_runs["PCA"].append((sdr_p, sir_p, sar_p))
        for s_idx in range(S):
            per_run_rows["PCA"].append({"run": run_idx, "source": s_idx+1, "SDR": float(sdr_p[s_idx]), "SIR": float(sir_p[s_idx]), "SAR": float(sar_p[s_idx])})

        # NMF (mono-STFT, align to reference)
        S_nmf_raw, _ = run_nmf(y_mix, n_components=n_components, seed=seed, stft=STFT_CFG, base_cfg=BASE_CFG, pb_desc=f"run {run_idx}")
        sdr_n, sir_n, sar_n, perm_n, S_nmf_aligned = evaluate_and_align(S_true, S_nmf_raw)
        LOGGER.log("[NMF] SDR: " + ", ".join(f"{v:.2f}" for v in sdr_n))
        LOGGER.log("[NMF] SIR: " + ", ".join(f"{v:.2f}" for v in sir_n))
        LOGGER.log("[NMF] SAR: " + ", ".join(f"{v:.2f}" for v in sar_n))
        metrics_runs["NMF"].append((sdr_n, sir_n, sar_n))
        for s_idx in range(S):
            per_run_rows["NMF"].append({"run": run_idx, "source": s_idx+1, "SDR": float(sdr_n[s_idx]), "SIR": float(sir_n[s_idx]), "SAR": float(sar_n[s_idx])})

        # Save & play separated audios for the representative run, in your requested order
        if EXP_CFG.save_wavs_for_run is not None and run_idx == EXP_CFG.save_wavs_for_run:
            run_dir = os.path.join(OUTPUT_DIR, f"run_{run_idx:02d}"); ensure_dir(run_dir)

            # Determine indices to show in the chosen order relative to reference order
            # Reference order is SOURCE_SHORT; disp_order reorders to SEPARATED_SHORT
            for out_idx, ref_idx in enumerate(disp_order, start=1):
                tag = SEPARATED_SHORT[out_idx-1]     # label string for display
                color = SEP_LABEL_COLOR[tag]         # use distinct colors for separated sources

                # FASTICA
                y_ica = S_ica_aligned[ref_idx]
                path = os.path.join(run_dir, f"separated_fastica_{out_idx}.wav")
                if SAVE_PRIVATE_AUDIO_FILES:
                    save_wav(path, y_ica, sr)
                sep_label = f"Separated Source {out_idx} ({tag})"
                print_signal_stats(y_ica, sr, sep_label)
                # metrics in the same ref_idx position
                LOGGER.log(f"SDR for {sep_label}: {metrics_runs['FastICA'][-1][0][ref_idx]:.4f}")
                LOGGER.log(f"SIR for {sep_label}: {metrics_runs['FastICA'][-1][1][ref_idx]:.4f}")
                LOGGER.log(f"SAR for {sep_label}: {metrics_runs['FastICA'][-1][2][ref_idx]:.4f}")
                play_audio(y_ica, sr, sep_label)
                if SHOW_PRIVATE_SIGNAL_FIGURES:
                    plot_waveform(y_ica, sr, f"Waveform of {sep_label}", color, label=sep_label)
                    plot_spectrogram(y_ica, sr, f"Spectrogram of {sep_label}", STFT_CFG)

                # PCA
                y_pca = S_pca_aligned[ref_idx]
                if SAVE_PRIVATE_AUDIO_FILES:
                    save_wav(os.path.join(run_dir, f"separated_pca_{out_idx}.wav"), y_pca, sr)

                # NMF
                y_nmf = S_nmf_aligned[ref_idx]
                if SAVE_PRIVATE_AUDIO_FILES:
                    save_wav(os.path.join(run_dir, f"separated_nmf_{out_idx}.wav"), y_nmf, sr)

    # 4) Aggregate stats and save
    summaries = {m: aggregate_mean_sd(metrics_runs[m]) for m in methods}
    LOGGER.log("\n=== Mean ± SD across runs (run-level means) ===")
    for m in methods:
        s = summaries[m]
        LOGGER.log(f"{m:8s}  SDR: {s['SDR'][0]:.2f} ± {s['SDR'][1]:.2f} | "
                   f"SIR: {s['SIR'][0]:.2f} ± {s['SIR'][1]:.2f} | "
                   f"SAR: {s['SAR'][0]:.2f} ± {s['SAR'][1]:.2f}  (n_runs={EXP_CFG.runs})")

    # Save per-run CSVs
    for m in methods:
        csv_path = os.path.join(OUTPUT_DIR, f"metrics_per_run_{m.lower()}.csv")
        save_csv(csv_path, per_run_rows[m], fieldnames=["run", "source", "SDR", "SIR", "SAR"])
        LOGGER.log(f"[Saved] {csv_path}")

    # Save config/environment snapshot
    env = detect_device_info()
    config = {
        "stft": asdict(STFT_CFG),
        "preprocess": asdict(PREP_CFG),
        "mix": asdict(MIX_CFG),
        "fastica": asdict(ICA_CFG),
        "baselines": asdict(BASE_CFG),
        "experiment": asdict(EXP_CFG),
        "private_audio_wav_export_enabled": SAVE_PRIVATE_AUDIO_FILES,
        "private_signal_figures_enabled": SHOW_PRIVATE_SIGNAL_FIGURES,
        "inline_audio_playback_enabled": ENABLE_INLINE_AUDIO_PLAYBACK,
        "environment": env,
        "input_basenames": [os.path.basename(path) for path in FILE_PATHS],
        "input_profile": PROFILE,
        "separated_display_order": SEPARATED_SHORT
    }
    save_json(os.path.join(OUTPUT_DIR, "config.json"), config)

    # Mean±SD table (run-level)
    table_rows = []
    for m in methods:
        table_rows.append({
            "method": m,
            "SDR_mean": summaries[m]["SDR"][0],
            "SDR_sd": summaries[m]["SDR"][1],
            "SIR_mean": summaries[m]["SIR"][0],
            "SIR_sd": summaries[m]["SIR"][1],
            "SAR_mean": summaries[m]["SAR"][0],
            "SAR_sd": summaries[m]["SAR"][1],
        })
    table_csv = os.path.join(OUTPUT_DIR, "metrics_summary_runlevel_sd_table.csv")
    save_csv(table_csv, table_rows, fieldnames=list(table_rows[0].keys()))
    LOGGER.log(f"[Saved] {table_csv}")

    # 5) Comparison plot (mean ± SD)
    if EXP_CFG.save_figs:
        barplot_compare(summaries, "BSS Comparison (FastICA vs PCA vs NMF)", outpath=os.path.join(OUTPUT_DIR, "bar_compare_mean_sd.png"))
        LOGGER.log(f"\n[Saved] {os.path.join(OUTPUT_DIR, 'bar_compare_mean_sd.png')}")

    # Wrap up
    LOGGER.log("\n=== Artifacts saved ===")
    LOGGER.log(f"- Text log: {LOG_PATH}")
    LOGGER.log(f"- Config: {os.path.join(OUTPUT_DIR, 'config.json')}")
    LOGGER.log(f"- Dataset overview: {os.path.join(OUTPUT_DIR, 'dataset_overview.csv')}")
    LOGGER.log(f"- Mixing matrix: {os.path.join(OUTPUT_DIR, 'mixing_matrix_A.csv')}")
    if SAVE_PRIVATE_AUDIO_FILES:
        LOGGER.log(f"- Private WAVs dir: {os.path.join(OUTPUT_DIR, 'wavs')}")
    if SAVE_PRIVATE_AUDIO_FILES and EXP_CFG.save_wavs_for_run is not None:
        LOGGER.log(f"- Private separated WAVs (rep run): {os.path.join(OUTPUT_DIR, f'run_{EXP_CFG.save_wavs_for_run:02d}')}")

if __name__ == "__main__":
    try:
        main()
    finally:
        LOGGER.close()
